#  天猫订单数据可视化

### 一、数据理解
本数据集共收集了天猫发生在2020年2月内的28010条数据。

共7个字段说明

1.订单编号：订单编号

2.总金额：订单总金额

3.买家实际支付金额：总金额 - 退款金额（在已付款的情况下）。金额为0（在未付款的情况下）

4.收货地址：各个省份

5.订单创建时间：下单时间

6.订单付款时间：付款时间

7.退款金额：付款后申请退款的金额。如无付过款，退款金额为0


### 二、分析目的
1.订单数在地图上的分布

2.每日订单数量分析（按照订单创建时间分组，得到每一天的订单量）

3.每小时订单数量分析

4.订单每个环节的转化转化率

### 三、数据分析与可视化过程

#### 1、导入需要的库、编码、路径设置

In [29]:
from pyecharts.globals import CurrentConfig, NotebookType
CurrentConfig.NOTEBOOK_TYPE = NotebookType.JUPYTER_LAB

import pandas as pd,os
import numpy as np
import matplotlib.pyplot as plt
import pyecharts.options as opts
from pyecharts.charts import Funnel as fu
from pyecharts.charts import Map as ma
from pyecharts.charts import Line
from pyecharts.charts import Bar
import warnings as wn
import seaborn as sns

plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False


#### 2、导入数据并进行预处理

订单付款时间存在缺失，预计是未付款订单，不做处理

In [ ]:
import pandas as pd
df = pd.read_csv('data/tmall_order_report.csv')
df.head()
df

,订单编号,总金额,买家实际支付金额,收货地址,订单创建时间,订单付款时间,退款金额
0,1,178.8,0.0,上海,2020-02-21 00:00:00,NaN,0.0
1,2,21.0,21.0,内蒙古自治区,2020-02-20 23:59:54,2020-02-21 00:00:02,0.0
2,3,37.0,0.0,安徽省,2020-02-20 23:59:35,NaN,0.0
3,4,157.0,157.0,湖南省,2020-02-20 23:58:34,2020-02-20 23:58:44,0.0
4,5,64.8,0.0,江苏省,2020-02-20 23:57:04,2020-02-20 23:57:11,64.8
...,...,...,...,...,...,...,...
28005,28006,37.0,37.0,四川省,2020-02-27 00:01:00,2020-02-27 00:01:10,0.0
28006,28007,69.0,0.0,上海,2020-02-27 00:00:18,NaN,0.0
28007,28008,69.0,0.0,上海,2020-02-27 00:00:17,NaN,0.0
28008,28009,37.0,37.0,辽宁省,2020-02-27 00:00:09,2020-02-27 00:00:17,0.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28010 entries, 0 to 28009
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   订单编号      28010 non-null  int64  
 1   总金额       28010 non-null  float64
 2   买家实际支付金额  28010 non-null  float64
 3   收货地址      28010 non-null  object 
 4   订单创建时间    28010 non-null  object 
 5   订单付款时间    24087 non-null  object 
 6   退款金额      28010 non-null  float64
dtypes: float64(3), int64(1), object(3)
memory usage: 1.5+ MB


#### 重复值

In [4]:
df.duplicated().sum()


0

#### 缺失值

In [5]:
df.isnull().sum()   # 订单付款时间 有2923个缺失值，属于正常现象，说明这些单位付过款，无需处理


订单编号           0
总金额            0
买家实际支付金额       0
收货地址           0
订单创建时间         0
订单付款时间      3923
退款金额           0
dtype: int64

#### 去掉属性列名的空格

In [6]:
df.columns


Index(['订单编号', '总金额', '买家实际支付金额', '收货地址 ', '订单创建时间', '订单付款时间 ', '退款金额'], dtype='object')

In [7]:
#发现有字段中有 空格
df.rename(columns=(lambda i:i.strip()),inplace=True)

### 3、各省份的订单量地图可视化

In [8]:
df.收货地址.value_counts()

上海          3353
广东省         2463
江苏省         2126
浙江省         2061
北京          2054
四川省         2019
山东省         1804
辽宁省         1187
天津          1153
湖南省         1099
河北省         1083
重庆          1036
河南省          966
云南省          778
安徽省          609
陕西省          536
福建省          489
山西省          465
广西壮族自治区      436
江西省          411
吉林省          401
黑龙江省         379
贵州省          345
内蒙古自治区       215
海南省          178
甘肃省          167
湖北省           75
新疆维吾尔自治区      58
宁夏回族自治区       42
青海省           19
西藏自治区          3
Name: 收货地址, dtype: int64

In [9]:
df.订单付款时间.notnull()  #查着订单付款时间不为空的
df[df.订单付款时间.notnull()]   #筛选没有付款的订单数据

,订单编号,总金额,买家实际支付金额,收货地址,订单创建时间,订单付款时间,退款金额
1,2,21.0,21.0,内蒙古自治区,2020-02-20 23:59:54,2020-02-21 00:00:02,0.0
3,4,157.0,157.0,湖南省,2020-02-20 23:58:34,2020-02-20 23:58:44,0.0
4,5,64.8,0.0,江苏省,2020-02-20 23:57:04,2020-02-20 23:57:11,64.8
5,6,327.7,148.9,浙江省,2020-02-20 23:56:39,2020-02-20 23:56:53,178.8
6,7,357.0,357.0,天津,2020-02-20 23:56:36,2020-02-20 23:56:40,0.0
...,...,...,...,...,...,...,...
28002,28003,77.0,77.0,重庆,2020-02-27 00:02:39,2020-02-27 00:03:27,0.0
28003,28004,157.0,157.0,山东省,2020-02-27 00:01:42,2020-02-27 00:01:47,0.0
28005,28006,37.0,37.0,四川省,2020-02-27 00:01:00,2020-02-27 00:01:10,0.0
28008,28009,37.0,37.0,辽宁省,2020-02-27 00:00:09,2020-02-27 00:00:17,0.0


In [10]:
def province_map(p):
    if p in ['北京','天津','上海','重庆']:
        return p +'市'
    return p
df.收货地址 = df.收货地址.map(province_map)

In [11]:
df.收货地址.value_counts()

上海市         3353
广东省         2463
江苏省         2126
浙江省         2061
北京市         2054
四川省         2019
山东省         1804
辽宁省         1187
天津市         1153
湖南省         1099
河北省         1083
重庆市         1036
河南省          966
云南省          778
安徽省          609
陕西省          536
福建省          489
山西省          465
广西壮族自治区      436
江西省          411
吉林省          401
黑龙江省         379
贵州省          345
内蒙古自治区       215
海南省          178
甘肃省          167
湖北省           75
新疆维吾尔自治区      58
宁夏回族自治区       42
青海省           19
西藏自治区          3
Name: 收货地址, dtype: int64

In [12]:
result = df[df.订单付款时间.notnull()].groupby('收货地址')[['订单编号']].count()  #统计各个省的订单量
result

,订单编号
收货地址,
上海市,3060
云南省,667
内蒙古自治区,176
北京市,1853
吉林省,336
四川省,1752
天津市,1031
宁夏回族自治区,40
安徽省,528


In [13]:
result2 = result.to_dict()['订单编号']
result2

{'上海市': 3060,
 '云南省': 667,
 '内蒙古自治区': 176,
 '北京市': 1853,
 '吉林省': 336,
 '四川省': 1752,
 '天津市': 1031,
 '宁夏回族自治区': 40,
 '安徽省': 528,
 '山东省': 1484,
 '山西省': 395,
 '广东省': 2022,
 '广西壮族自治区': 353,
 '新疆维吾尔自治区': 43,
 '江苏省': 1845,
 '江西省': 331,
 '河北省': 885,
 '河南省': 792,
 '浙江省': 1822,
 '海南省': 156,
 '湖北省': 57,
 '湖南省': 935,
 '甘肃省': 132,
 '福建省': 425,
 '西藏自治区': 2,
 '贵州省': 286,
 '辽宁省': 1012,
 '重庆市': 896,
 '陕西省': 441,
 '青海省': 18,
 '黑龙江省': 312}

In [14]:
list(result2.items())

[('上海市', 3060),
 ('云南省', 667),
 ('内蒙古自治区', 176),
 ('北京市', 1853),
 ('吉林省', 336),
 ('四川省', 1752),
 ('天津市', 1031),
 ('宁夏回族自治区', 40),
 ('安徽省', 528),
 ('山东省', 1484),
 ('山西省', 395),
 ('广东省', 2022),
 ('广西壮族自治区', 353),
 ('新疆维吾尔自治区', 43),
 ('江苏省', 1845),
 ('江西省', 331),
 ('河北省', 885),
 ('河南省', 792),
 ('浙江省', 1822),
 ('海南省', 156),
 ('湖北省', 57),
 ('湖南省', 935),
 ('甘肃省', 132),
 ('福建省', 425),
 ('西藏自治区', 2),
 ('贵州省', 286),
 ('辽宁省', 1012),
 ('重庆市', 896),
 ('陕西省', 441),
 ('青海省', 18),
 ('黑龙江省', 312)]

In [32]:
import pyecharts.options as opts
from pyecharts.charts import Map
map1 = (
    Map()
    .add(
         "各省份订单数",
        list(result2.items()),
        'china',
        is_map_symbol_show = False
    )
    .set_global_opts(title_opts = opts.TitleOpts(title = "各省份订单量"),
        visualmap_opts=opts.VisualMapOpts(max_ = 3060)
    )   
    
    
)
map1.load_javascript()


In [33]:
map1.render_notebook()

分析：从图中可以看出，上海、广东、北京、江浙及四川的订单量普遍较高。

### 4、每日订单数量分析（时间序列分析）
#### 按照订单创建时间分组，得到每一天的订单量 

时间列格式为object，需要修改为datetime

In [18]:
df['订单创建时间']=pd.to_datetime(df['订单创建时间'])
df['订单付款时间']=pd.to_datetime(df['订单付款时间'])
df[df.订单付款时间.notnull()]

,订单编号,总金额,买家实际支付金额,收货地址,订单创建时间,订单付款时间,退款金额
1,2,21.0,21.0,内蒙古自治区,2020-02-20 23:59:54,2020-02-21 00:00:02,0.0
3,4,157.0,157.0,湖南省,2020-02-20 23:58:34,2020-02-20 23:58:44,0.0
4,5,64.8,0.0,江苏省,2020-02-20 23:57:04,2020-02-20 23:57:11,64.8
5,6,327.7,148.9,浙江省,2020-02-20 23:56:39,2020-02-20 23:56:53,178.8
6,7,357.0,357.0,天津市,2020-02-20 23:56:36,2020-02-20 23:56:40,0.0
...,...,...,...,...,...,...,...
28002,28003,77.0,77.0,重庆市,2020-02-27 00:02:39,2020-02-27 00:03:27,0.0
28003,28004,157.0,157.0,山东省,2020-02-27 00:01:42,2020-02-27 00:01:47,0.0
28005,28006,37.0,37.0,四川省,2020-02-27 00:01:00,2020-02-27 00:01:10,0.0
28008,28009,37.0,37.0,辽宁省,2020-02-27 00:00:09,2020-02-27 00:00:17,0.0


In [19]:
df.订单创建时间

0       2020-02-21 00:00:00
1       2020-02-20 23:59:54
2       2020-02-20 23:59:35
3       2020-02-20 23:58:34
4       2020-02-20 23:57:04
                ...        
28005   2020-02-27 00:01:00
28006   2020-02-27 00:00:18
28007   2020-02-27 00:00:17
28008   2020-02-27 00:00:09
28009   2020-02-27 00:00:06
Name: 订单创建时间, Length: 28010, dtype: datetime64[ns]

In [20]:
order_add_time = df.订单创建时间.map(lambda x: x.strftime('%Y-%m-%d'))

In [21]:
result3 = df.groupby(order_add_time)[['订单编号']].count().to_dict()['订单编号']  #按照订单创建时间分组，得到每一天的订单量
result3

{'2020-02-01': 176,
 '2020-02-02': 222,
 '2020-02-03': 267,
 '2020-02-04': 469,
 '2020-02-05': 369,
 '2020-02-06': 144,
 '2020-02-07': 177,
 '2020-02-09': 404,
 '2020-02-10': 27,
 '2020-02-11': 15,
 '2020-02-12': 1,
 '2020-02-13': 5,
 '2020-02-14': 7,
 '2020-02-15': 5,
 '2020-02-17': 390,
 '2020-02-18': 1015,
 '2020-02-19': 1025,
 '2020-02-20': 1345,
 '2020-02-21': 2068,
 '2020-02-22': 2027,
 '2020-02-23': 2200,
 '2020-02-24': 1998,
 '2020-02-25': 3416,
 '2020-02-26': 2849,
 '2020-02-27': 2586,
 '2020-02-28': 2691,
 '2020-02-29': 2112}

In [22]:
line = (
    Line()
    .add_xaxis(list(result3.keys()))
    .add_yaxis(
        '订单量',
        list(result3.values())
       )
    .set_global_opts( 
        title_opts = opts.TitleOpts(title = "每天订单量"),
        yaxis_opts = opts.AxisOpts(
              splitline_opts = opts.SplitLineOpts(is_show = True)
        
        )
    )
    #标签配置项
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show= False ),
        markpoint_opts= opts.MarkPointOpts(
            data=[
                opts.MarkPointItem(type_='max')
            ]
        )
    )
                     
      
)
line. render_notebook()

#### 每日的订单量最高在2月25日，达到3416，2月10至2月14日，订单量很低，几乎为0。

### 5、每小时订单量统计可视化

In [23]:
order_add_time2 = df.订单创建时间.map(lambda x: x.strftime('%H'))
order_add_time2.value_counts()

21    2204
22    2126
20    1990
15    1720
10    1672
23    1644
11    1640
19    1532
14    1500
16    1481
12    1367
13    1339
09    1283
18    1270
17    1205
00    1043
08     880
07     556
01     530
02     341
06     250
03     189
04     135
05     113
Name: 订单创建时间, dtype: int64

In [24]:
result4 = df.groupby(order_add_time2)[['订单编号']].count().to_dict()['订单编号']
result4

{'00': 1043,
 '01': 530,
 '02': 341,
 '03': 189,
 '04': 135,
 '05': 113,
 '06': 250,
 '07': 556,
 '08': 880,
 '09': 1283,
 '10': 1672,
 '11': 1640,
 '12': 1367,
 '13': 1339,
 '14': 1500,
 '15': 1720,
 '16': 1481,
 '17': 1205,
 '18': 1270,
 '19': 1532,
 '20': 1990,
 '21': 2204,
 '22': 2126,
 '23': 1644}

In [25]:
bar = (
    Bar()
    .add_xaxis(list(result4.keys()))
    .add_yaxis(
        '订单量',
        list(result4.values())
       )
    .set_global_opts( 
        title_opts = opts.TitleOpts(title = "每小时订单量"),
        yaxis_opts = opts.AxisOpts(
              splitline_opts = opts.SplitLineOpts(is_show = True)
        
        )
    )
    #标签配置项
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show= False ),
        markpoint_opts= opts.MarkPointOpts(
            data=[
                opts.MarkPointItem(type_='max')
            ]
        )
    )
                     
      
)
bar.render_notebook()

#### 分析：根据每小时订单量柱状图可以得出：凌晨3~5点，下单的人数较少，晚上9点的订单量最多。

### 6、订单转化率分析
根据上述信息，无法确定分析方向，所以假设2月总销售目标是220万，为了进一步了解数据，可以从不同环节转化率来看看是哪个环节出了问题。

用户行为转化路径：创建--付款--实付--全额

这里采用绝对转化率实现：1.求出各环节的订单数；2.求转化率：用本环节的订单数除以订单创建数


In [26]:
# 看各个环节的漏斗------创建--付款--实际--全额 每一层的转化是多少
rates=pd.Series({'下单':df['订单创建时间'].count(),
'付款':df['订单付款时间'].count(),
'实付款':df[df['买家实际支付金额']>0].shape[0],  #巧妙运算  df[df['买家实际支付金额']>0].shape(0)，shape(0) 将返回数据的行数。
'全额付款':df[df['买家实际支付金额']==df['总金额']].shape[0]},name='订单量').to_frame()  #to_frame(): 将字典转换为DataFrame。
 
rates
 
rates['转化率']=rates['订单量'].apply(lambda x:round(x/rates.iloc[0,0],3)) #添加
rates['相对转化率']=round((rates/rates.shift())['订单量'].fillna(1),3)  #添加
 
print(rates)
 
#展示漏斗图
c=(fu().add('',[list(z) for z in zip(rates.index,rates['转化率'])],
            label_opts=opts.LabelOpts(position='inside',formatter='{b}:{c}'))).set_global_opts(title_opts=opts.TitleOpts(title='整体转化率（%）'))
c.render_notebook()

        订单量    转化率  相对转化率
下单    28010  1.000  1.000
付款    24087  0.860  0.860
实付款   18955  0.677  0.787
全额付款  18441  0.658  0.973


#### 分析：根据TrustData的报告显示，淘宝2015年平时的订单成功率为97.4%。而本次分析的付款转化率约86%低于预期标准，实际成交及全额成交的环节转化率不到70%。属于比较低的水平。

### 四、总结
1、由订单数在地图上的分布来看，上海、广东、北京、江浙及四川的订单量普遍较高。上海市为全国最高，达到3060。

2、每日的订单量最高在2月25日，达到3416，2月10至2月14日，订单量很低，几乎为0。而根据每小时订单量统计可以得出：凌晨3~5点，下单的人数较少，晚上9点的订单量最多。

3、转化率情况：付款成功率为86%，低于正常水平的97.4%，实际成交和全额成交的转化率分别为67.7%和65.8%。